In [11]:
# en -> zh
def print_translations():
    with open("./en-zh.en-filtered.en.subword.test.desubword", 'r', encoding='utf-8') as en_file, \
         open("./zh.translated.desubword", 'r', encoding='utf-8') as zh_file, \
         open("./en-zh.zh-filtered.zh.subword.test.desubword", encoding="utf-8") as zh_correct_file:
        
        for en_line, zh_line, zh_correct_line in zip(en_file, zh_file, zh_correct_file):
            en_line = en_line.strip()
            zh_line = zh_line.strip()
            zh_correct_line = zh_correct_line.strip()
            
            print(f"English:            {en_line}")
            print(f"Chinese:            {zh_line}")
            print(f"Chinese (correct):  {zh_correct_line}")
            print("-" * 50)

print_translations()

English:            These worsening indices of health care or health studies in Africa demand a new look. We cannot keep on doing things the way we've always done them.
Chinese:            非洲医疗保险和健康研究的 越来越糟, 需要一个新眼光。我们不能一直 以自己的方式做这些事情。
Chinese (correct):  非洲越来越低的医疗保健指数和越来越少人关注的医疗研究状况 必须改变,我们不能一直 停滞不前
--------------------------------------------------
English:            Sorry about that.
Chinese:            不好意思。
Chinese (correct):  很抱歉.
--------------------------------------------------
English:            And then this is a picture of one of the most famous hospitals in America.
Chinese:            这是美国最著名的医院之一
Chinese (correct):  看看这幅画吧, 这是美国最著名的医院之一,
--------------------------------------------------
English:            Nextpedition turns the trip into a game, with surprising twists and turns along the way.
Chinese:            然后把旅行变成游戏, 一路上都有意想不到的转折。
Chinese (correct):  就把旅行变成游戏 一路上都有意想不到的惊奇
--------------------------------------------------
English:            It's a fantastic de

In [ ]:
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.metrics.distance import edit_distance
from collections import defaultdict
import pandas as pd
import jieba

nltk.download('punkt')

In [46]:
def analyze_translations_meteor(en_path, zh_trans_path, zh_correct_path):
    """
    Analyzes translations using METEOR and edit distance metrics.

    Parameters:
    - en_path: Path to the English source sentences.
    - zh_trans_path: Path to the translated Chinese sentences.
    - zh_correct_path: Path to the reference Chinese translations.

    Returns:
    - df_sentences: DataFrame containing sentence-level analysis.
    - df_mismatches: DataFrame containing word mismatch counts.
    """
    sentence_results = []  # To store sentence-level metrics
    word_mismatches = defaultdict(int)  # To tally missing/mistranslated words
    smoothing = SmoothingFunction().method1  # Smoothing for sentence-level BLEU

    with open(en_path, 'r', encoding='utf-8') as en_file, \
         open(zh_trans_path, 'r', encoding='utf-8') as zh_file, \
         open(zh_correct_path, 'r', encoding='utf-8') as zh_correct_file:
        
        for idx, (en_line, zh_line, zh_correct_line) in enumerate(zip(en_file, zh_file, zh_correct_file)):
            en_line = en_line.strip()
            zh_line = zh_line.strip()
            zh_correct_line = zh_correct_line.strip()
            
            # Tokenize the Chinese sentences
            zh_tokens = list(jieba.cut(zh_line))
            zh_correct_tokens = list(jieba.cut(zh_correct_line))

            # Compute the sentence-level scores.
            meteor = meteor_score([zh_correct_tokens], zh_tokens)
            bleu = sentence_bleu([zh_correct_tokens], zh_tokens, smoothing_function=smoothing)
            dist = edit_distance(zh_line, zh_correct_line)
            
            sentence_results.append({
                'English': en_line,
                'Chinese (model)': zh_line,
                'Chinese (correct)': zh_correct_line,
                'METEOR': meteor,
                'BLEU': bleu,
                'EditDistance': dist
            })
            
            # Tally missing words from the reference that do not appear in the model output
            for word in zh_correct_tokens:
                if word not in zh_tokens:
                    word_mismatches[word] += 1
                    
    df_sentences = pd.DataFrame(sentence_results)
    df_mismatches = pd.DataFrame(list(word_mismatches.items()), columns=['Word', 'MismatchCount'])
    df_mismatches.sort_values(by='MismatchCount', ascending=False, inplace=True)
    
    return df_sentences, df_mismatches


In [47]:
en_path = "./en-zh.en-filtered.en.subword.test.desubword"
zh_trans_path = "./zh.translated.desubword"
zh_correct_path = "./en-zh.zh-filtered.zh.subword.test.desubword"

df_sentences, df_mismatches = analyze_translations_meteor(en_path, zh_trans_path, zh_correct_path)

### Sort by METEOR (asc), edit distance (desc)

Edit distance is used as well to highlight sentences that are semantically different

In [71]:
df_sentences.sort_values(by=['METEOR', 'EditDistance'], ascending=[True, False]).head(10)

,English,Chinese (model),Chinese (correct),METEOR,BLEU,EditDistance
587,"You make a PowerPoint, you know?","你制作了Powerpoint,你们知道吗?",我们需要Power Point,0.0,0.0,13
1539,To her girlfriends she said that.,女友说她这么说。,”我知道我自己在做什么“,0.0,0.0,12
361,That's an amazing thing.,这是件令人惊叹的事。,很神奇吧!,0.0,0.0,10
741,It's transformational.,它是一种转型。,这将带来伟大的变革.,0.0,0.0,10
1226,You can see this one.,你可以看到这个。,"看着这怪兽,从左到右",0.0,0.0,10
1589,This is Paldin.,这是Palindin。,"这位是宝丁,",0.0,0.0,10
1219,The textile industry is incredibly mobile.,"纺织业非常灵活,",纺织品行业极具移动性。,0.0,0.0,8
184,Audience: Yes! Yeah!,观众:没错!,"是的,观众们。",0.0,0.0,7
417,So the upshot was this.,截图就是这个,这是结果。,0.0,0.0,6
740,Us.,美,是我们自己。,0.0,0.0,6


### Sort sentences based on input length

In [67]:
df_sentences.sort_values(by='English', key=lambda x: x.str.len()).head(10)

,English,Chinese (model),Chinese (correct),METEOR,BLEU,EditDistance
740,Us.,美,是我们自己。,0.000000,0.000000,6
563,7.5.,7.5次。,7.5次。,0.981481,0.562341,0
591,Mmm.,嗯,嗯,0.500000,0.177828,0
855,Thanks.,谢谢。,谢谢。,0.937500,0.316228,0
552,E: Yar.,爱因斯坦:,爱因斯坦:是。,0.493421,0.116334,2
1414,Me too.,我也是。,我也是。,0.992188,1.000000,0
1350,Stop it.,停止吧,停下吧,0.250000,0.149535,1
1082,Namaste.,马斯特。,谢谢。,0.250000,0.149535,3
1746,Yes? OK.,是吗?好的。,愿意?好的,0.701058,0.202052,3
578,Owl. Owl.,猫头鹰。猫头鹰。,猫头鹰。猫头鹰。,0.992188,1.000000,0


In [74]:
print("\nMost Frequently Mistranslated (Missing) Words:")
df_mismatches.head()


Most Frequently Mistranslated (Missing) Words:


,Word,MismatchCount
90,的,209
26,是,183
25,了,138
18,",",126
64,在,125


### Result findings
- Mistranslated Entity
- Not capturing the correct sense
  - e.g And in summer, here, killer wasps.
  - e.g 法律人员 vs 合法的人 in "Humans and legal persons are not synonymous."
- Cannot translate well if 1 token for input (?)
  - e.g Us.;美;	是我们自己。
- Captured sentences literally (without considering enough context) 
  - Not good because the model is suppose to look at the entire corpus (?)
  - e.g It was all about going for the center.
- Wrong inversion
  - e.g Nobody wants to buy a mini well when they buy a car.
- Inaccuracy in capturing the correct sense
  - e.g It's a mind setting.
- Wrong intensity of adjective
  - e.g I was less exotic in this Whitopia.